In [251]:
import pandas as pd

df_dm_city = pd.read_csv(r"C:\Users\Vikas\Desktop\Good cabs analysis project\RPC13_Input_For_Participants\datasets\csv_files\dim_city.csv")
df_fact_trips = pd.read_csv(r"C:\Users\Vikas\Desktop\Good cabs analysis project\RPC13_Input_For_Participants\datasets\csv_files\fact_trips.csv")
df_dim_repeat_trip_distribution = pd.read_csv(r"C:\Users\Vikas\Desktop\Good cabs analysis project\RPC13_Input_For_Participants\datasets\csv_files\dim_repeat_trip_distribution.csv")
monthly_target_trips_df = pd.read_csv(r"C:\Users\Vikas\Desktop\Good cabs analysis project\RPC13_Input_For_Participants\datasets\csv_files\monthly_target_trips.csv")
monthly_target_new_passengers_df = pd.read_csv(r"C:\Users\Vikas\Desktop\Good cabs analysis project\RPC13_Input_For_Participants\datasets\csv_files\monthly_target_new_passengers.csv")
fact_passenger_summary_df = pd.read_csv(r"C:\Users\Vikas\Desktop\Good cabs analysis project\RPC13_Input_For_Participants\datasets\csv_files\fact_passenger_summary.csv")
city_target_passenger_rating = pd.read_csv(r"C:\Users\Vikas\Desktop\Good cabs analysis project\RPC13_Input_For_Participants\datasets\csv_files\city_target_passenger_rating.csv")

In [252]:
#Top 3 cities by total trips over the entire analysis period.
merged_df = pd.merge(df_dm_city, df_fact_trips, on='city_id', how='inner')


grouped_df = merged_df.groupby('city_name').agg(total_trips=('trip_id', 'count')).reset_index()


top_3_cities = grouped_df.sort_values(by='total_trips', ascending=False).head(3)

print(top_3_cities)

  city_name  total_trips
3    Jaipur        76888
5   Lucknow        64299
7     Surat        54843


In [253]:
#bottom 3 cities by total trips over the entire analysis period

merged_df = pd.merge(df_dm_city, df_fact_trips, on='city_id', how='inner')


grouped_df = merged_df.groupby('city_name').agg(total_trips=('trip_id', 'count')).reset_index()


bottom_3_cities = grouped_df.sort_values(by='total_trips', ascending=True).head(3)

print(bottom_3_cities)

       city_name  total_trips
6         Mysore        16238
1     Coimbatore        21104
9  Visakhapatnam        28366


In [254]:
#Avg fair per trip per km by city , top city
merged_df = pd.merge(df_dm_city, df_fact_trips, on='city_id', how='inner')


result = merged_df.groupby('city_name').apply(
    lambda x: (x['fare_amount'].sum() * 1.0) / x['distance_travelled(km)'].sum()
).reset_index(name='avg_fare_per_km_by_city')


top_city = result.sort_values(by='avg_fare_per_km_by_city', ascending=False).head(1)

print(top_city)

  city_name  avg_fare_per_km_by_city
3    Jaipur                 16.11818


In [255]:
#Avg fair per trip per km by city , bottom city

merged_df = pd.merge(df_dm_city, df_fact_trips, on='city_id', how='inner')


result = merged_df.groupby('city_name').apply(
    lambda x: (x['fare_amount'].sum() * 1.0) / x['distance_travelled(km)'].sum()
).reset_index(name='avg_fare_per_km_by_city')


bottom_city = result.sort_values(by='avg_fare_per_km_by_city', ascending=True).head(1)

print(bottom_city)

  city_name  avg_fare_per_km_by_city
8  Vadodara                10.294225


In [256]:
#Avg ratings by city and passenger type , top city
merged_df = pd.merge(df_dm_city, df_fact_trips, on='city_id', how='inner')


filtered_df = merged_df[merged_df['passenger_type'] == 'new']


grouped_df = (
    filtered_df
    .groupby('city_name', as_index=False)
    .agg(avg_passenger_rating=('passenger_rating', 'mean'))
)


sorted_df = grouped_df.sort_values(by='avg_passenger_rating', ascending=False)


top_city = sorted_df.head(1)

print(top_city)

  city_name  avg_passenger_rating
4     Kochi              8.987394


In [257]:
#Avg ratings by city and passenger type = new, bottom city
merged_df = pd.merge(df_dm_city, df_fact_trips, on='city_id', how='inner')


filtered_df = merged_df[merged_df['passenger_type'] == 'new']


grouped_df = (
    filtered_df
    .groupby('city_name', as_index=False)
    .agg(avg_passenger_rating=('passenger_rating', 'mean'))
)


sorted_df = grouped_df.sort_values(by='avg_passenger_rating', ascending=True)


bottom_city = sorted_df.head(1)

print(top_city)

  city_name  avg_passenger_rating
4     Kochi              8.987394


In [258]:
#Avg ratings by city and passenger type = repeated , true city
merged_df = pd.merge(df_dm_city, df_fact_trips, on='city_id', how='inner')


filtered_df = merged_df[merged_df['passenger_type'] == 'repeated']


grouped_df = (
    filtered_df
    .groupby('city_name', as_index=False)
    .agg(avg_passenger_rating=('passenger_rating', 'mean'))
)


sorted_df = grouped_df.sort_values(by='avg_passenger_rating', ascending=False)


top_city = sorted_df.head(1)

print(top_city)

  city_name  avg_passenger_rating
4     Kochi              8.003665


In [259]:
#Avg ratings by city and passenger type = repeated , bottom city
merged_df = pd.merge(df_dm_city, df_fact_trips, on='city_id', how='inner')


filtered_df = merged_df[merged_df['passenger_type'] == 'repeated']


grouped_df = (
    filtered_df
    .groupby('city_name', as_index=False)
    .agg(avg_passenger_rating=('passenger_rating', 'mean'))
)


sorted_df = grouped_df.sort_values(by='avg_passenger_rating', ascending=True)


bottom_city = sorted_df.head(1)

print(bottom_city)

  city_name  avg_passenger_rating
8  Vadodara              5.978629


In [260]:
#Peak and low demand months by city
merged = pd.merge(df_dm_city, df_fact_trips, on='city_id')


merged['month'] = pd.to_datetime(merged['date']).dt.month
monthly_distance = merged.groupby(['city_name', 'month'])['distance_travelled(km)'].sum().reset_index()
monthly_distance.rename(columns={'distance_travelled(km)': 'total_distance_in_month'}, inplace=True)

monthly_distance['peak_rank'] = monthly_distance.groupby('city_name')['total_distance_in_month']\
                                                .rank(method='dense', ascending=False)

peak_demand = monthly_distance[monthly_distance['peak_rank'] == 1][['city_name', 'month']]
peak_demand.rename(columns={'month': 'peak_demand_month'}, inplace=True)


monthly_distance['low_rank'] = monthly_distance.groupby('city_name')['total_distance_in_month']\
                                               .rank(method='dense', ascending=True)

lowest_demand = monthly_distance[monthly_distance['low_rank'] == 1][['city_name', 'month']]
lowest_demand.rename(columns={'month': 'lowest_demand_month'}, inplace=True)


final_result = pd.merge(peak_demand, lowest_demand, on='city_name')

print(final_result)

       city_name  peak_demand_month  lowest_demand_month
0     Chandigarh                  2                    4
1     Coimbatore                  4                    6
2         Indore                  5                    6
3         Jaipur                  2                    6
4          Kochi                  5                    6
5        Lucknow                  2                    5
6         Mysore                  5                    1
7          Surat                  4                    1
8       Vadodara                  4                    6
9  Visakhapatnam                  4                    6


In [261]:
# Weekday vs Weekend trip demand by city

# Add weekday and month columns
df_fact_trips['date'] = pd.to_datetime(df_fact_trips['date'])
df_fact_trips['month'] = df_fact_trips['date'].dt.month
df_fact_trips['weekday'] = df_fact_trips['date'].dt.weekday + 1  # To match SQL's 1=Sunday, 7=Saturday

# Filter for Jan to June (1 to 6) and weekdays (Mon to Fri = 2 to 6)
weekday_trips = df_fact_trips[
    (df_fact_trips['month'].between(1, 6)) &
    (df_fact_trips['weekday'].between(2, 6))
]

# Join with dim_city
cte = weekday_trips.merge(df_dm_city, on='city_id') \
    .groupby(['city_name', 'month', 'weekday']) \
    .agg(trips=('trip_id', 'count')) \
    .reset_index()

cte1 = cte.groupby(['city_name', 'month']) \
    .agg(total_weekdays_trips=('trips', 'sum')) \
    .reset_index()

# Filter for Jan to June (1 to 6) and weekends (Sun or Sat = 1 or 7)
weekend_trips = df_fact_trips[
    (df_fact_trips['month'].between(1, 6)) &
    (df_fact_trips['weekday'].isin([1, 7]))
]

# Join with dim_city
cte2 = weekend_trips.merge(df_dm_city, on='city_id') \
    .groupby(['city_name', 'month', 'weekday']) \
    .agg(trips=('trip_id', 'count')) \
    .reset_index()

cte3 = cte2.groupby(['city_name', 'month']) \
    .agg(total_weekends_trips=('trips', 'sum')) \
    .reset_index()

final_df = cte1.merge(cte3, on=['city_name', 'month'])

final_df

,city_name,month,total_weekdays_trips,total_weekends_trips
0,Chandigarh,1,4336,2474
1,Chandigarh,2,4783,2604
2,Chandigarh,3,4323,2246
3,Chandigarh,4,3548,2018
4,Chandigarh,5,4466,2154
5,Chandigarh,6,3924,2105
6,Coimbatore,1,2461,1190
7,Coimbatore,2,2310,1094
8,Coimbatore,3,2519,1161
9,Coimbatore,4,2439,1222


In [262]:
df_dm_city

,city_id,city_name
0,RJ01,Jaipur
1,UP01,Lucknow
2,GJ01,Surat
3,KL01,Kochi
4,MP01,Indore
5,CH01,Chandigarh
6,GJ02,Vadodara
7,AP01,Visakhapatnam
8,TN01,Coimbatore
9,KA01,Mysore


In [263]:
#
# dim_city and dim_repeat_trip_distribution

# First CTE: cte
cte = (
    df_dm_city.merge(df_dim_repeat_trip_distribution, on='city_id', how='inner')
    .groupby(['city_name', 'trip_count'], as_index=False)
    .agg(total_repeat_passeneger_count=('repeat_passenger_count', 'sum'))
)

# Second CTE: cte1
cte1 = (
    df_dm_city.merge(df_dim_repeat_trip_distribution, on='city_id', how='inner')
    .groupby('city_name', as_index=False)
    .agg(total_repeat_passeneger_count=('repeat_passenger_count', 'sum'))
)

# Final join and calculation
final_df = (
    cte.merge(cte1, on='city_name', suffixes=('_x', '_y'))
    .assign(
        percentage_repeat_passenger_distribution=lambda df:
        (df['total_repeat_passeneger_count_x'] * 100.0 / df['total_repeat_passeneger_count_y']).round(2)
    )
    .loc[:, ['city_name', 'trip_count', 'percentage_repeat_passenger_distribution']]
    .sort_values('city_name')
)


print(final_df)

        city_name trip_count  percentage_repeat_passenger_distribution
0      Chandigarh   10-Trips                                      1.79
1      Chandigarh    2-Trips                                     32.31
2      Chandigarh    3-Trips                                     19.25
3      Chandigarh    4-Trips                                     15.74
4      Chandigarh    5-Trips                                     12.21
..            ...        ...                                       ...
84  Visakhapatnam    4-Trips                                      9.98
85  Visakhapatnam    5-Trips                                      5.44
86  Visakhapatnam    6-Trips                                      3.19
87  Visakhapatnam    7-Trips                                      1.98
89  Visakhapatnam    9-Trips                                      0.88

[90 rows x 3 columns]


In [264]:
monthly_target_trips_df

,month,city_id,total_target_trips
0,2024-03-01,MP01,7000
1,2024-05-01,KA01,2500
2,2024-04-01,UP01,11000
3,2024-02-01,GJ02,6000
4,2024-05-01,KL01,9000
5,2024-02-01,UP01,13000
6,2024-01-01,AP01,4500
7,2024-01-01,CH01,7000
8,2024-02-01,KL01,7500
9,2024-03-01,UP01,13000


In [265]:
df_fact_trips['month'] = df_fact_trips['date'].dt.month

cte = (
    df_dm_city.merge(df_fact_trips, on='city_id', how='inner')
    .groupby(['city_name', 'month', 'city_id'], as_index=False)
    .agg(total_trips_in_month=('trip_id', 'count'))
)

# Step 2: Create cte1 by joining with monthly_target_trips
monthly_target_trips_df.rename(columns={'month': 'date'}, inplace=True)
monthly_target_trips_df['date'] = pd.to_datetime(monthly_target_trips_df['date'])
monthly_target_trips_df['month'] = monthly_target_trips_df['date'].dt.month

cte1 = (
    cte.merge(monthly_target_trips_df, on=['city_id', 'month'], how='inner')
    [['city_name', 'month', 'total_trips_in_month', 'total_target_trips']]
)

# Step 3: Create cte2 with percentage difference and target met status
cte2 = cte1.copy()
cte2['percentage_diff'] = (
    (cte2['total_trips_in_month'] - cte2['total_target_trips']) * 100.0
    / cte2['total_trips_in_month']
)
cte2['whether_target_met'] = cte2['total_target_trips'] <= cte2['total_trips_in_month']
cte2['whether_target_met'] = cte2['whether_target_met'].astype(int)  # convert boolean to 0/1

# Final Step: Select city_name where all values of whether_target_met are the same
result = (
    cte2.groupby('city_name')['whether_target_met']
    .nunique()
    .reset_index()
    .query('whether_target_met == 1')
    [['city_name']]
)

print(result)

  city_name
3    Jaipur
5   Lucknow
6    Mysore
8  Vadodara


In [266]:
fact_passenger_summary_df.rename(columns={'month': 'date'}, inplace=True)
fact_passenger_summary_df['date'] = pd.to_datetime(fact_passenger_summary_df['date'])
fact_passenger_summary_df['month'] = fact_passenger_summary_df['date'].dt.month  # Extract month
cte = (
    df_dm_city
    .merge(fact_passenger_summary_df, on='city_id', how='inner')
    .groupby(['city_name', 'month', 'city_id'], as_index=False)
    .agg({'new_passengers': 'sum'})
)

# Step 2: Join with monthly_target_new_passengers on city_id and month
monthly_target_new_passengers_df.rename(columns={'month': 'date'}, inplace=True)
monthly_target_new_passengers_df['date'] = pd.to_datetime(monthly_target_new_passengers_df['date'])
monthly_target_new_passengers_df['month'] = monthly_target_new_passengers_df['date'].dt.month  # Extract month
cte1 = cte.merge(
    monthly_target_new_passengers_df,
    left_on=['city_id', 'month'],
    right_on=['city_id', 'month'],
    how='inner'
)

# Step 3: Calculate percentage_diff and whether_target_met
cte1['percentage_diff'] = ((cte1['new_passengers'] - cte1['target_new_passengers']) * 100.0) / cte1['new_passengers']
cte1['whether_target_met'] = cte1.apply(
    lambda row: '0' if row['target_new_passengers'] > row['new_passengers'] else '1',
    axis=1
)

# Step 4: Group by city_name and filter cities where whether_target_met is the same for all months
result = (
    cte1.groupby('city_name')['whether_target_met']
    .nunique()
    .reset_index()
)
final_cities = result[result['whether_target_met'] == 1]['city_name']

print(final_cities)

1    Coimbatore
2        Indore
Name: city_name, dtype: object


In [267]:
cte = (
    df_dm_city
    .merge(df_fact_trips, on='city_id', how='inner')
    .groupby(['city_name', 'city_id'], as_index=False)
    .agg(avg_passenger_rating=('passenger_rating', 'mean'))
)

# Step 2: Join with city_target_passenger_rating
cte1 = cte.merge(
    city_target_passenger_rating,
    on='city_id',
    how='inner'
)

# Step 3: Calculate percentage_diff and whether_target_met
cte1['percentage_diff'] = ((cte1['avg_passenger_rating'] - cte1['target_avg_passenger_rating']) * 100.0) / cte1['avg_passenger_rating']
cte1['whether_target_met'] = cte1.apply(
    lambda row: 0 if row['target_avg_passenger_rating'] > row['avg_passenger_rating'] else 1,
    axis=1
)

# Step 4: Keep cities where whether_target_met is consistent (all 0s or all 1s)
result = (
    cte1.groupby('city_name')['whether_target_met']
    .nunique()
    .reset_index()
)
final_cities = result[result['whether_target_met'] == 1]['city_name']

print(final_cities)

0       Chandigarh
1       Coimbatore
2           Indore
3           Jaipur
4            Kochi
5          Lucknow
6           Mysore
7            Surat
8         Vadodara
9    Visakhapatnam
Name: city_name, dtype: object


In [268]:


# Merge both tables on city_id
merged_df = df_dm_city.merge(fact_passenger_summary_df, on='city_id', how='inner')

#: Calculate repeat_passenger_percentage
merged_df['repeat_passenger_percentage'] = (merged_df['repeat_passengers'] * 100.0) / merged_df['total_passengers']

#: Group by city_name and month, calculate average percentage
grouped = (
    merged_df
    .groupby(['city_name', 'month'], as_index=False)
    .agg(avg_repeat_passenger_percentage=('repeat_passenger_percentage', 'mean'))
)

# Sort descending and select top 2
top_2 = grouped.sort_values(by='avg_repeat_passenger_percentage', ascending=False).head(2)

print(top_2)

   city_name  month  avg_repeat_passenger_percentage
46     Surat      5                        49.922288
47     Surat      6                        49.174917


In [269]:
# Merge both tables on city_id
merged_df = df_dm_city.merge(fact_passenger_summary_df, on='city_id', how='inner')

#: Calculate repeat_passenger_percentage
merged_df['repeat_passenger_percentage'] = (merged_df['repeat_passengers'] * 100.0) / merged_df['total_passengers']

#: Group by city_name and month, calculate average percentage
grouped = (
    merged_df
    .groupby(['city_name', 'month'], as_index=False)
    .agg(avg_repeat_passenger_percentage=('repeat_passenger_percentage', 'mean'))
)

# Sort descending and select top 2
top_2 = grouped.sort_values(by='avg_repeat_passenger_percentage', ascending=True).head(2)

print(top_2)

   city_name  month  avg_repeat_passenger_percentage
37    Mysore      2                         7.991266
36    Mysore      1                         8.078910


In [270]:
df1 = df_dm_city.merge(fact_passenger_summary_df, on='city_id', how='inner')
df1['repeat_passenger_rate'] = (df1['repeat_passengers'] * 100.0) / df1['total_passengers']
cte = (
    df1.groupby('city_name', as_index=False)
    .agg(avg_repeat_passenger_rate=('repeat_passenger_rate', 'mean'))
)

# --- Step 2: CTE1 - Calculate avg_trip_rate and avg_passenger_rate for repeated passengers ---
df2 = df_dm_city.merge(df_fact_trips, on='city_id', how='inner')

# Filter for repeated passengers
df2_repeated = df2[df2['passenger_type'] == 'repeated'].copy()
df2_repeated['trip_rate'] = df2_repeated['fare_amount'] / df2_repeated['distance_travelled(km)']

cte1 = (
    df2_repeated.groupby('city_name', as_index=False)
    .agg(
        avg_trip_rate=('trip_rate', 'mean'),
        avg_passenger_rate=('passenger_rating', 'mean')
    )
)

# --- Step 3: Final join and sort ---
final_df = cte.merge(cte1, on='city_name', how='inner')
final_df = final_df.sort_values(by='avg_repeat_passenger_rate', ascending=False)

print(final_df)

       city_name  avg_repeat_passenger_rate  avg_trip_rate  avg_passenger_rate
7          Surat                  42.963123      10.786333            5.995511
5        Lucknow                  38.131873      12.005993            5.985741
2         Indore                  32.957829      10.877183            7.473961
8       Vadodara                  30.793795      10.426154            5.978629
9  Visakhapatnam                  28.811272      12.225145            7.989628
1     Coimbatore                  23.671272      11.011525            7.475457
4          Kochi                  22.376614      13.811553            8.003665
0     Chandigarh                  21.750782      11.826070            7.493798
3         Jaipur                  18.329207      15.653994            7.991042
6         Mysore                  11.208195      14.627041            7.978495
